In [51]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp,"martin2009assessing")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "martinordas2009assessing_assessing generalization_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [52]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="martin2009assessing"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [53]:
df.rename(columns={"column1": "ape",
    "species":"species_x",
    "sex":"sex_y"}, inplace=True)

In [54]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

In [55]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df.columns
df.rename(columns={"ape": "participant",
                   "age":"age_original"}, inplace=True)

df['age_original'].replace(np.nan, 0, inplace=True)
# df['age_original'].unique()

In [56]:

df['age_in_years'] = (df['age_original'])//12

df['age_original'].replace( 0, np.nan,inplace=True)
df['age_in_years'].replace( 0, np.nan,inplace=True)

In [57]:
replace_list_1 = [[1,'functional_trap'],
                [2,'fake_trap'],
                [3,'painted_trap']]
for x, y in replace_list_1:
    df['condition'].replace( x, y,inplace=True)



In [58]:

replace_list_2 = [['l','left'],
                  ['l       ','left'],
                  ['r       ','right'],
                ['r','right']]
df['trap_side'] = df['trap_side'].astype(str)
for x, y in replace_list_2:
    df['trap_side'].replace( x, y,inplace=True)

df['trap_side'].unique()

array(['left', 'right'], dtype=object)

In [59]:

martin2009assessing_standardized=df[['study_id','participant', 'age_original','age_in_years','sex', 'species', 
                                     'session', 'trial',   'condition',
      'trap_side', 'correct']]
comp_out_path_stand = os.path.join(out_pathway, 'martin2009assessing_exp1_standardized.csv')
martin2009assessing_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =martin2009assessing_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
martin2009assessing_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'martin2009assessing_exp1_glossary.csv')
martin2009assessing_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
